# LunarLander — a first trained agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s9-lunarlander-starter.ipynb)

**MS2A — Machine Learning Practice · Session 9, Reinforcement Learning**

Land a lander between the two flags, in strong wind: [challenge 43](https://ml-arena.com/viewchallenge/43)
on ML-Arena, ranked by the **mean episode return**.

| Section | What you do |
|---|---|
| 0 | Install, connect to ML-Arena |
| 1 | Look at the environment: spaces, reward, what "solved" means |
| 2 | Score a random agent: the floor |
| 3 | **Train a linear policy with the cross-entropy method** (a few minutes on CPU) |
| 4 | Write `agent.py`, check it locally, submit it |
| 5 | Questions, and where to go next |

## 0. Setup

In [ ]:
!pip install -q "gymnasium[box2d]==1.3.0" "mlarena-sdk>=4.6"

import time
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
CHALLENGE_ID = 43

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. The environment

The challenge does not use the default lander: it adds **wind and turbulence**. Train on
exactly the configuration the leaderboard runs, or your scores will not transfer.

In [ ]:
ENV_KWARGS = dict(continuous=False, gravity=-9.0, enable_wind=True,
                  wind_power=15.0, turbulence_power=1.7)   # the challenge's env.py
env = gym.make("LunarLander-v3", **ENV_KWARGS)

print("observation:", env.observation_space)
print("actions:    ", env.action_space)
obs, info = env.reset(seed=0)
print("first observation:", np.round(obs, 3))

**Observation (8 numbers):** position $x, y$, velocity $v_x, v_y$, angle, angular velocity,
and two booleans: left / right leg touching the ground.

**Actions (4):** 0 do nothing, 1 fire left engine, 2 fire main engine, 3 fire right engine.

**Reward:** shaped at every step — closer to the pad, slower and more level is better; each
leg contact +10; firing the main engine −0.3 per frame, a side engine −0.03; then **+100**
for landing at rest or **−100** for a crash. An episode scoring **≥ 200** counts as *solved*.
An episode ends on landing, crash, leaving the screen, or after 1,000 steps.

## 2. A random agent: the floor

`evaluate` plays `n` episodes with a policy (a function `obs -> action`) and returns the
mean return. Fixed seeds make two policies comparable on the same episodes.

In [ ]:
def run_episode(policy, seed):
    obs, _ = env.reset(seed=seed)
    total, done = 0.0, False
    while not done:
        obs, reward, terminated, truncated, _ = env.step(policy(obs))
        total += reward
        done = terminated or truncated
    return total

def evaluate(policy, n=20, seed=10_000):
    return np.mean([run_episode(policy, seed + i) for i in range(n)])

random_policy = lambda obs: env.action_space.sample()
print(f"random agent: {evaluate(random_policy):.1f}")

## 3. A first training: linear policy + cross-entropy method

The policy is a **linear score per action**, and the agent plays the best one:

$$a = \arg\max_{a}\,(o^\top W + b)_a, \qquad W \in \mathbb{R}^{8\times 4},\; b \in \mathbb{R}^{4}$$

That is 36 numbers $\theta$. The **cross-entropy method (CEM)** searches them without
gradients:

1. sample a population of $\theta \sim \mathcal{N}(\mu, \sigma^2)$;
2. score each one on a few episodes;
3. keep the best ones (the *elite*), and refit $\mu, \sigma$ to them; repeat.

No value function, no backpropagation: only "try, keep what works".

In [ ]:
def linear_policy(theta):
    W, b = theta[:32].reshape(8, 4), theta[32:]
    return lambda obs: int(np.argmax(obs @ W + b))

N_ITER, POP, N_ELITE, N_EPISODES = 60, 50, 10, 5   # TODO: try other values
rng = np.random.default_rng(0)
mu, sigma = np.zeros(36), np.ones(36)
history, best_theta, best_score = [], mu, -np.inf

start = time.time()
for it in range(N_ITER):
    thetas = mu + sigma * rng.standard_normal((POP, 36))
    seeds = rng.integers(0, 1_000_000, N_EPISODES)        # same episodes for every candidate
    scores = np.array([np.mean([run_episode(linear_policy(t), int(s)) for s in seeds])
                       for t in thetas])
    elite = thetas[np.argsort(scores)[-N_ELITE:]]          # TODO: what if N_ELITE = 1? = POP?
    mu, sigma = elite.mean(axis=0), elite.std(axis=0) + 0.05
    history.append(scores.mean())
    if it % 5 == 4:                                        # keep the best mean policy seen
        score = evaluate(linear_policy(mu))
        if score > best_score:
            best_theta, best_score = mu.copy(), score
        print(f"iter {it + 1:2d}  population mean {scores.mean():7.1f}  "
              f"policy mu {score:7.1f}  ({time.time() - start:.0f}s)")

In [ ]:
plt.plot(history)
plt.xlabel("CEM iteration"); plt.ylabel("mean return of the population")
plt.show()

print(f"best policy, 50 held-out episodes: {evaluate(linear_policy(best_theta), n=50, seed=50_000):.1f}")
np.save("theta.npy", best_theta)

## 4. Submit

The platform imports `Agent` from `agent.py` and calls `setup(observation_space,
action_space)` once, then `choose_action(observation, ...)` at every step. Return `None` once
the episode is over. The weights travel as a second file, `theta.npy`, read from the
folder of `agent.py`. Only `numpy` is needed at inference.

In [ ]:
%%writefile agent.py
from pathlib import Path

import numpy as np


class Agent:
    def __init__(self):
        theta = np.load(Path(__file__).parent / "theta.npy")
        self.W, self.b = theta[:32].reshape(8, 4), theta[32:]

    def setup(self, observation_space, action_space):
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return int(np.argmax(np.asarray(observation) @ self.W + self.b))

In [ ]:
# Local check: the file you upload, on 10 episodes.
from agent import Agent

agent = Agent()
print(f"agent.py locally: {evaluate(lambda obs: agent.choose_action(obs), n=10):.1f}")

In [ ]:
result = client.submit(CHALLENGE_ID, files=["agent.py", "theta.npy"],
                       submission_name="cem-linear", wait=True)
print(result["status"])

In [ ]:
client.leaderboard(CHALLENGE_ID, top=10)

## 5. Questions

1. **Exploration.** Where does exploration happen in CEM? What does the `+ 0.05` on $\sigma$
   prevent, and what happens to the search without it?
2. **Noise.** Each candidate is scored on 5 windy episodes. Why does the population mean in
   the plot go up and down? What do `N_EPISODES` and the shared `seeds` change?
3. **Sample efficiency.** Count the environment episodes used by the training. Compare with
   tabular Q-learning: what does CEM *not* learn from inside an episode?
4. **The plateau.** Why can a linear function of the 8 observations not express "fire the
   main engine only when falling fast *and* close to the ground"?
5. **Next step.** Train a DQN or PPO (`stable-baselines3`, Session 10) on the same
   `ENV_KWARGS`, save the model, load it in `Agent.__init__`, and submit with a matching
   runtime: `client.submit(..., runtime={"language": "python", "framework": "torch"})`.